# 04 — Full benchmark on Parkinson's: all methods, no shift (k = 0) and shift (k = 1, …, 7)
Adds two further baselines to the methods of notebooks 01–02:

* **OTCP** — Klein et al. (2026): entropic OT map of the residual vectors to the spherical uniform
  distribution, score = transported radius, three-way split (train / hold-out / calibration).
* **GRCP** — Conrad et al. (2026): conditional spatial rank of the residual vector, rank-first kernel
  estimator (their Eq. 16), score = rank norm.

Both use the deep network's mean as predictor, so they differ from Deep-LocalCov only in the score.
Like Braun et al., they assume exchangeable data (no shift correction).

Shift design as notebook 02 (Gan & Liu Sec. 5, 1000 distinct test records). The ablations
(Algorithm 2 literal, oracle ratio) stay in notebook 02.

Volume = region area in UPDRS²: per split, the median over test points (GRCP: over a random subset
of test points, computed on a grid), then the **median over splits** (robust to the occasional
very large region; the mean over splits is also saved).

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import experiment as ex
import deep

print("device:", deep.DEVICE)

## Configuration

In [ ]:
QUICK = False

CFG = dict(
    SEED=2026, ALPHA=0.10, N_TRAIN=1000, N_TEST=1000, N_REPS=20, B=1000, SCP_FIT_FRAC=0.5,
    K_VALUES=[0, 1, 2, 3, 4, 5, 6, 7],
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3,
    OT_M=1024,             # [my choice] reference points (Klein et al. Fig. 7: larger m is better)
    OT_EPS=0.1,            # [paper] Klein et al. Sec. 4.2: eps = 0.1 relative to the cost scale
    GRID_SIDE=150,         # [my choice] grid for OTCP / GRCP region areas
    GRCP_VOL_POINTS=100,   # [my choice] test inputs used for the GRCP area
)
CC = dict(WSC_DELTA=0.10, WSC_N_DIR=1000, WSC_GRID=50, CEC_J=10)
if QUICK:
    CFG.update(N_REPS=1, B=200, MAX_EPOCHS=40, K_VALUES=[0, 4, 7], GRCP_VOL_POINTS=20)
    CC.update(WSC_N_DIR=100)

OUT_DIR = "../results/benchmark_parkinsons"
os.makedirs(OUT_DIR, exist_ok=True)
CFG

## Data and run

In [ ]:
df, X_all, Y_all = cp.load_parkinsons("../data/parkinsons_updrs.data")
sampler = cp.ShiftSampler(X_all)
res_df = ex.run(X_all, Y_all, sampler, CFG, CC, deep)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Tables

In [ ]:
by_k, tables = ex.summarize(res_df)
by_k.to_csv(f"{OUT_DIR}/summary_by_k.csv", index=False)
for name, tab in tables.items():
    tab.to_csv(f"{OUT_DIR}/table_{name}_by_k.csv")
    print(f"\n{name.upper()} by k\n", tab.to_string())

main_k = [0, 4, 7] if 4 in CFG['K_VALUES'] else CFG['K_VALUES']
compact = pd.concat({f"k={k}": by_k[by_k.k == k].set_index('method')[['coverage', 'volume', 'WSC']]
                     for k in main_k}, axis=1).reindex(ex.METHODS)
compact.to_csv(f"{OUT_DIR}/table_main.csv")
print("\nMAIN TABLE (k = 0 / 4 / 7)\n", compact.to_string())
print("\nOTCP / GRCP regions reaching the grid edge (fraction):\n",
      by_k[by_k.method.isin(['OTCP', 'GRCP'])].groupby('method').grid_edge_hit.mean())

## Figure: coverage, volume and WSC versus k

In [ ]:
style = {'SCP-L2-LR': ('tab:gray', 's', '--'), 'WCP-L2-LR': ('tab:brown', 's', '-'),
         'EU': ('tab:pink', 'o', '--'), 'EUc-LR': ('tab:red', 'o', '-'),
         'OTCP': ('tab:purple', 'v', '--'), 'GRCP': ('tab:olive', 'P', '--'),
         'Deep-LocalCov': ('tab:cyan', '^', '--'),
         'Deep-LocalCov + WCP (ours)': ('tab:green', '^', '-'),
         'Deep-LocalCov + EUc (ours)': ('tab:blue', 'D', '-')}
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for m in ex.METHODS:
    s = by_k[by_k.method == m]
    c, mk, ls = style[m]
    ax[0].plot(s.k, s.coverage, color=c, marker=mk, ls=ls, label=m)
    ax[1].plot(s.k, s.volume, color=c, marker=mk, ls=ls)
    ax[2].plot(s.k, s.WSC, color=c, marker=mk, ls=ls)
ax[0].axhline(100 * (1 - CFG['ALPHA']), color='k', lw=1, ls=':')
ax[0].set_ylabel('coverage (%)'); ax[1].set_ylabel('region area (UPDRS$^2$)'); ax[2].set_ylabel('WSC (%)')
ax[1].set_yscale('log')
for x in ax:
    x.set_xlabel('shift intensity $k$')
fig.legend(loc='lower center', ncol=5, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.1))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_benchmark_curves.png", dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))
print("CONFIG:", CFG, "\nCC:", CC)